# Qdrant Cloud Vector Store

This notebook demonstrates FFAI's Qdrant backend connected to a **remote
Qdrant Cloud cluster**.

- Requires `QDRANT_CLUSTER_ENDPOINT` and `QDRANT_KEY` in your `.env` file
- Creates a unique collection, indexes documents, searches, then **deletes
  the collection** at the end
- No data is left on the cluster after the notebook finishes

Uses synthetic embeddings (random unit vectors) to avoid needing an
embedding API key.

<div class="page-break"></div>

---

## Section 1: Setup and Credentials

In [ ]:
import os
import sys
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from dotenv import load_dotenv  # noqa: E402

load_dotenv()

CLUSTER_URL = os.getenv('QDRANT_CLUSTER_ENDPOINT', '')
API_KEY = os.getenv('QDRANT_KEY', '')

if not CLUSTER_URL or not API_KEY:
    raise RuntimeError(
        'Set QDRANT_CLUSTER_ENDPOINT and QDRANT_KEY in .env '
        'to run this notebook.'
    )

print(f'Cluster: {CLUSTER_URL}')
print(f'API key: {API_KEY[:8]}...')

<div class="page-break"></div>

---

## Section 2: Create the Cloud Store

Pass `url=` and `api_key=` to connect to the cloud cluster. We use a unique
collection name with a timestamp to avoid collisions.

In [ ]:
import time

import numpy as np

from ffai.rag.stores import get_store

DIM = 128
COLLECTION = f'nb_demo_{int(time.time())}'

store = get_store(
    backend='qdrant',
    collection_name=COLLECTION,
    embedding_dim=DIM,
    url=CLUSTER_URL,
    api_key=API_KEY,
)

print(f'Backend: {store.name}')
print(f'Collection: {COLLECTION}')
print(f'Initial count: {store.count()}')

<div class="page-break"></div>

---

## Section 3: Add Documents

Index sample documents with synthetic embeddings.

In [ ]:
def make_embedding(text: str, dim: int = DIM) -> list[float]:
    rng = np.random.default_rng(hash(text) & 0xFFFFFFFF)
    vec = rng.standard_normal(dim)
    return (vec / np.linalg.norm(vec)).tolist()


import asyncio  # noqa: E402

from examples._rag_data.download import download  # noqa: E402
from ffai.rag.splitters import get_chunker  # noqa: E402

sample_docs = download()
total = 0
for name, path in sample_docs.items():
    text = path.read_text(encoding='utf-8')
    chunks = get_chunker('recursive', chunk_size=500, chunk_overlap=50).chunk(text)
    ids = [f'{name}_chunk{i}' for i in range(len(chunks))]
    texts = [c.content for c in chunks]
    embs = [make_embedding(t) for t in texts]
    metas = [{'source': name, 'chunking_strategy': 'recursive'} for _ in chunks]
    added = asyncio.run(store.aadd(ids, texts, embs, metas))
    total += added
    print(f'  {name}: {added} chunks')

print(f'\nTotal indexed: {total} chunks')
print(f'Sources: {store.list_sources()}')

<div class="page-break"></div>

---

## Section 4: Search

Query the cloud collection with and without metadata filters.

In [ ]:
queries = [
    ('async programming', None),
    ('authentication token', {'source': 'api_docs.md'}),
]

for qtext, where in queries:
    emb = make_embedding(qtext)
    hits = asyncio.run(store.asearch(emb, top_k=3, where=where))
    filter_str = f' (filter: {where})' if where else ''
    print(f'Query: "{qtext}"{filter_str}')
    for i, h in enumerate(hits, 1):
        print(f'  {i}. [{h.score:.4f}] {h.content[:80]}...')
        print(f'     source={h.source}')
    print()

<div class="page-break"></div>

---

## Section 5: Cleanup — Delete the Collection

Remove the collection from the cloud cluster so nothing is left behind.

In [ ]:
from qdrant_client import QdrantClient

admin = QdrantClient(url=CLUSTER_URL, api_key=API_KEY)
admin.delete_collection(COLLECTION)

collections = [c.name for c in admin.get_collections().collections]
remaining = [c for c in collections if c == COLLECTION]

print(f'Deleted collection: {COLLECTION}')
print(f'Collection still exists: {len(remaining) > 0}')
print('\nCleanup complete. Zero collections left on cluster from this notebook.')